# Lección 5: Introducción a Machine Learning Escalable (Spark MLlib)  
## Proyecto: Retail Analytics Pipeline – RetailMax

## 1. Introducción

En esta etapa final se implementan modelos de Machine Learning utilizando Spark MLlib para analizar el comportamiento de los usuarios en la plataforma e-commerce.

Se desarrollan modelos supervisados y no supervisados que permiten clasificar usuarios y segmentarlos, generando información útil para la toma de decisiones en marketing.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("RetailAnalyticsPipeline") \
    .master("local[*]") \
    .getOrCreate()


# Dataset principal
df = spark.read.csv("data/ecommerce_client_data.csv", header=True, inferSchema=True)

# Dataset agregado (resultado de la lección anterior)
metrics_df = spark.read.csv("data/visitor_metrics.csv", header=True, inferSchema=True)

df.show(5)
metrics_df.show()

+--------------+-----------------------+-------------+----------------------+--------------+-----------------------+-----------+---------+----------+----------+-----+----------------+-------+------+-----------+-----------------+-------+-------+
|Administrative|Administrative_Duration|Informational|Informational_Duration|ProductRelated|ProductRelated_Duration|BounceRates|ExitRates|PageValues|SpecialDay|Month|OperatingSystems|Browser|Region|TrafficType|      VisitorType|Weekend|Revenue|
+--------------+-----------------------+-------------+----------------------+--------------+-----------------------+-----------+---------+----------+----------+-----+----------------+-------+------+-----------+-----------------+-------+-------+
|             0|                    0.0|            0|                   0.0|             1|                    0.0|        0.2|      0.2|       0.0|       0.0|  Feb|               1|      1|     1|          1|Returning_Visitor|  false|  false|
|             0|    

## 3. Integración de métricas agregadas

Se incorporan las métricas generadas previamente (valor promedio por tipo de visitante) al dataset principal, enriqueciendo la información disponible para los modelos.

In [ ]:
from pyspark.sql.functions import col

df = df.join(metrics_df, on="VisitorType", how="left")
df.select("VisitorType", "avg_page_value").show(5)

+-----------------+-----------------+
|      VisitorType|   avg_page_value|
+-----------------+-----------------+
|Returning_Visitor|5.006175700140938|
|Returning_Visitor|5.006175700140938|
|Returning_Visitor|5.006175700140938|
|Returning_Visitor|5.006175700140938|
|Returning_Visitor|5.006175700140938|
+-----------------+-----------------+
only showing top 5 rows


## 4. Preparación de datos

In [8]:
# Indexación
from pyspark.ml.feature import StringIndexer

# Revenue a string antes de indexar
df = df.withColumn("Revenue_str", col("Revenue").cast("string"))

# Variable categórica
label_indexer = StringIndexer(inputCol="Revenue_str", outputCol="label")
df = label_indexer.fit(df).transform(df)


## 5. Creación de features

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "VisitorType_index",
    "avg_page_value"   
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df = assembler.transform(df)

## 6. División de datos

In [10]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

## 7. Modelo supervisado: Regresión Logística

In [11]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")

model = lr.fit(train_df)

## 8. Evaluación del modelo

In [12]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

predictions = model.transform(test_df)
evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions)

print("AUC:", auc)

AUC: 0.88292787395285




Se utilizó el área bajo la curva ROC (AUC) como métrica principal, ya que permite evaluar la capacidad del modelo para discriminar entre usuarios que convierten y los que no.

Esta métrica es especialmente relevante en contextos donde las clases pueden estar desbalanceadas, como suele ocurrir en problemas de conversión.

## Predicciones

In [13]:
predictions.select("label", "prediction", "probability").show(10)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  0.0|       0.0|[0.99577944992647...|
|  0.0|       0.0|[0.99577944992647...|
|  0.0|       0.0|[0.99577944992647...|
|  0.0|       0.0|[0.99577944992647...|
|  0.0|       0.0|[0.99577944992647...|
|  0.0|       0.0|[0.99577944992647...|
|  0.0|       0.0|[0.99576998641357...|
|  0.0|       0.0|[0.96766061713143...|
|  0.0|       0.0|[0.92854806309501...|
|  0.0|       0.0|[0.90783748193667...|
+-----+----------+--------------------+
only showing top 10 rows


## 9. Modelo no supervisado: K-Means

In [14]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(k=3, seed=42)
kmeans_model = kmeans.fit(df)
clusters = kmeans_model.transform(df)

## Segmentación de usuarios

Se aplicó K-Means para identificar grupos de usuarios con comportamientos similares. Este enfoque permite descubrir patrones no evidentes y segmentar clientes sin necesidad de una variable objetivo.

La segmentación obtenida puede ser utilizada para personalizar estrategias de marketing y mejorar la experiencia del usuario.

In [15]:
clusters.select("prediction", "VisitorType", "avg_page_value").show(10)

+----------+-----------------+-----------------+
|prediction|      VisitorType|   avg_page_value|
+----------+-----------------+-----------------+
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
|         0|Returning_Visitor|5.006175700140938|
+----------+-----------------+-----------------+
only showing top 10 rows


## 10. Evaluación del clustering

In [17]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(clusters)

print("Silhouette score:", silhouette)

Silhouette score: 0.887236995414713


## 11. Insights para marketing

El análisis de las métricas agregadas revela diferencias significativas entre los tipos de visitantes:

- El segmento "Other" presenta el mayor valor promedio por sesión (~18.19)
- Los "New_Visitor" tienen un valor intermedio (~10.77)
- Los "Returning_Visitor" muestran el menor valor (~5.00)

La incorporación de esta información como feature en los modelos permite mejorar la capacidad predictiva y entender mejor el comportamiento de los usuarios.

El modelo de clasificación permite identificar usuarios con mayor probabilidad de conversión, mientras que el clustering permite segmentar usuarios en grupos con características similares.

Aplicaciones:

- Personalización de campañas de marketing
- Identificación de usuarios de alto valor
- Optimización de estrategias de retención

## Insight clave

El valor promedio por tipo de visitante muestra diferencias significativas entre segmentos, lo que valida la necesidad de estrategias diferenciadas de marketing.

La combinación de análisis exploratorio y modelos de Machine Learning permite transformar estos patrones en decisiones accionables.

## 12. Conclusión

Se desarrolló un pipeline completo de Machine Learning utilizando Spark MLlib, integrando datos procesados y métricas agregadas para mejorar el análisis.

El uso de features derivadas (como el valor promedio por tipo de visitante) permitió enriquecer los modelos y generar insights más relevantes para el negocio.

Este enfoque demuestra la capacidad de integrar Big Data y Machine Learning en un entorno escalable, permitiendo transformar datos en decisiones estratégicas.